# 🛠️ 02 - Preprocessing & Feature Engineering
Notebook này thực hiện:
1. Xử lý giá trị thiếu (Missing values) và ngoại lai (Outliers)
2. Tạo các đặc trưng mới (Feature Engineering): Product Popularity, CLV, Seasonal trends, Marketing
3. Mã hóa biến phân loại và chuẩn hóa biến số
4. Phân chia Train/Test (Temporal vs Random Split) và xuất dữ liệu ra `data/processed/`

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse

# Tìm thư mục gốc dù Jupyter được mở từ thư mục dự án hay thư mục notebooks.
ROOT_DIR = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "README.md").exists()),
    Path.cwd().parent,
)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src import config
from src.data.builder import build_product_week_panel
from src.data.feature_engineering import engineer_product_week_features
from src.data.preprocessing import build_weekly_preprocessor, split_product_week_data

## 1. Tải dữ liệu thô (Raw Data)

In [ ]:
# Luôn dựng lại panel từ dữ liệu nguồn để tránh dùng nhầm kết quả cũ sau khi thay đổi quy tắc xử lý.
weekly_panel = build_product_week_panel()

print(f"Product-week rows: {len(weekly_panel):,}")
print(f"Distinct products: {weekly_panel['product_id'].nunique():,}")
print(f"Calendar weeks: {weekly_panel['week_start'].nunique():,}")

## 2. Tiền xử lý dữ liệu (Missing Values & Outliers)

In [ ]:
# Kiểm tra giá trị khuyết trước khi huấn luyện. Không cắt ngưỡng doanh thu vì đây
# là biến mục tiêu thực tế và cần được giữ nguyên phân phối.
missing_report = weekly_panel.isna().sum().sort_values(ascending=False)
display(missing_report[missing_report.gt(0)].rename("missing_count").to_frame())
print(f"Target minimum: {weekly_panel[config.WEEKLY_TARGET_COLUMN].min():.2f} BRL")
print(f"Target maximum: {weekly_panel[config.WEEKLY_TARGET_COLUMN].max():.2f} BRL")
print(f"Missing category rows (will use train mode): {weekly_panel['category'].isna().sum():,}")

# Kiểm tra tính duy nhất của khóa, tuần bắt đầu vào thứ Hai và chuỗi tuần liên tục.
duplicate_keys = int(weekly_panel.duplicated(["product_id", "week_start"]).sum())
monday_start_violations = int(weekly_panel["week_start"].dt.dayofweek.ne(0).sum())
week_counts = weekly_panel.groupby("product_id", sort=False)["week_start"].agg(["min", "max", "count"])
expected_week_counts = ((week_counts["max"] - week_counts["min"]).dt.days // 7) + 1
discontinuous_products = int(week_counts["count"].ne(expected_week_counts).sum())

# Đối soát doanh thu tổng với các dòng order item đã giao trong dữ liệu nguồn.
raw_orders = pd.read_csv(config.RAW_DATA_DIR / "olist_orders_dataset.csv", usecols=["order_id", "order_status"])
delivered_order_ids = raw_orders.loc[raw_orders["order_status"].eq("delivered"), "order_id"]
raw_items = pd.read_csv(config.RAW_DATA_DIR / "olist_order_items_dataset.csv", usecols=["order_id", "price"])
source_delivered_items = raw_items.loc[raw_items["order_id"].isin(delivered_order_ids)]
source_revenue = float(source_delivered_items["price"].sum())
panel_revenue = float(weekly_panel[config.WEEKLY_TARGET_COLUMN].sum())
revenue_difference = abs(source_revenue - panel_revenue)

quality_checks = pd.DataFrame([
    {"check": "Unique product-week key", "value": duplicate_keys, "expected": 0, "passed": duplicate_keys == 0},
    {"check": "Monday week start", "value": monday_start_violations, "expected": 0, "passed": monday_start_violations == 0},
    {"check": "Continuous weekly series per product", "value": discontinuous_products, "expected": 0, "passed": discontinuous_products == 0},
    {"check": "Delivered item rows represented", "value": len(source_delivered_items), "expected": int(weekly_panel["units_sold"].sum()), "passed": len(source_delivered_items) == int(weekly_panel["units_sold"].sum())},
    {"check": "Revenue reconciliation difference (BRL)", "value": revenue_difference, "expected": "<= 0.000001", "passed": revenue_difference <= 1e-6},
    {"check": "Negative revenue rows", "value": int(weekly_panel[config.WEEKLY_TARGET_COLUMN].lt(0).sum()), "expected": 0, "passed": weekly_panel[config.WEEKLY_TARGET_COLUMN].ge(0).all()},
    {"check": "Zero-sales product-week share", "value": weekly_panel[config.WEEKLY_TARGET_COLUMN].eq(0).mean(), "expected": "Reported; zero weeks are retained", "passed": True},
] + [
    {
        "check": f"Missing values: {column}",
        "value": int(weekly_panel[column].isna().sum()),
        "expected": "Imputed using train-only statistics" if column in config.WEEKLY_NUMERICAL_FEATURES + config.WEEKLY_CATEGORICAL_FEATURES else 0,
        "passed": column in config.WEEKLY_NUMERICAL_FEATURES + config.WEEKLY_CATEGORICAL_FEATURES or weekly_panel[column].notna().all(),
    }
    for column in weekly_panel.columns
])
display(quality_checks)
quality_checks.to_csv(config.WEEKLY_DATA_QUALITY_FILE, index=False)
if not quality_checks["passed"].all():
    raise ValueError("Weekly panel data-quality checks failed; review the displayed report.")
print("Không áp dụng capping ngoại lai; mọi kiểm tra chất lượng dữ liệu đều đạt.")

## 3. Tạo các đặc trưng mới (Feature Engineering theo đề bài)
- **Product Popularity**: reviews, ratings, popularity score
- **Customer Lifetime Value (CLV)**: RFM, AOV
- **Seasonal Trends**: month, quarter, holidays, peak season
- **Marketing**: ad spend ratio, discount rate

In [ ]:
# Đặc trưng trễ, trượt, đánh giá và lịch sử khách hàng chỉ dùng các tuần đã hoàn tất.
featured = engineer_product_week_features(weekly_panel)
featured.to_csv(config.FEATURED_WEEKLY_PANEL_FILE, index=False)
feature_columns = config.WEEKLY_NUMERICAL_FEATURES + config.WEEKLY_CATEGORICAL_FEATURES
print(f"Featured panel shape: {featured.shape}")
print(f"Numeric predictors: {len(config.WEEKLY_NUMERICAL_FEATURES)}")
print(f"Categorical predictors before encoding: {len(config.WEEKLY_CATEGORICAL_FEATURES)}")
display(featured[feature_columns + [config.WEEKLY_TARGET_COLUMN]].head())

## 4. Phân chia Train/Test Split (Tránh rò rỉ dữ liệu)

In [ ]:
# Chia theo tuần lịch hoàn chỉnh để mọi sản phẩm trong cùng tuần thuộc cùng một tập.
splits = split_product_week_data(featured)
for split_name, split_frame in splits.items():
    print(
        f"{split_name:>10}: {len(split_frame):,} rows | "
        f"{split_frame['week_start'].min():%Y-%m-%d} to "
        f"{split_frame['week_start'].max():%Y-%m-%d}"
    )

assert splits["train"]["week_start"].max() < splits["validation"]["week_start"].min()
assert splits["validation"]["week_start"].max() < splits["test"]["week_start"].min()

## 5. Lưu dữ liệu đã xử lý ra `data/processed/`

In [ ]:
# Chỉ học tham số điền khuyết và chuẩn hóa từ dữ liệu train trong quá khứ.
# Danh mục sản phẩm cung cấp 73 nhãn Olist; đây là siêu dữ liệu cấu trúc, không phải thống kê được fit.
known_categories = sorted(weekly_panel["category"].dropna().unique().tolist())
preprocessor = build_weekly_preprocessor(category_values=known_categories)
X_train_raw = splits["train"][feature_columns]
X_validation_raw = splits["validation"][feature_columns]
X_test_raw = splits["test"][feature_columns]
y_train = splits["train"][config.WEEKLY_TARGET_COLUMN].to_numpy(dtype=np.float64)
y_validation = splits["validation"][config.WEEKLY_TARGET_COLUMN].to_numpy(dtype=np.float64)
y_test = splits["test"][config.WEEKLY_TARGET_COLUMN].to_numpy(dtype=np.float64)

X_train = sparse.csr_matrix(preprocessor.fit_transform(X_train_raw), dtype=np.float32)
X_validation = sparse.csr_matrix(preprocessor.transform(X_validation_raw), dtype=np.float32)
X_test = sparse.csr_matrix(preprocessor.transform(X_test_raw), dtype=np.float32)

# Lưu ma trận sparse để tối ưu dung lượng cho dữ liệu one-hot hơn một triệu dòng.
for cache_path, matrix in [
    (config.CACHE_X_TRAIN, X_train),
    (config.CACHE_X_VAL, X_validation),
    (config.CACHE_X_TEST, X_test),
]:
    sparse.save_npz(cache_path, matrix)
np.save(config.CACHE_Y_TRAIN, y_train)
np.save(config.CACHE_Y_VAL, y_validation)
np.save(config.CACHE_Y_TEST, y_test)
feature_names = preprocessor.get_feature_names_out().tolist()
config.CACHE_FEATURE_NAMES.write_text(json.dumps(feature_names, indent=2), encoding="utf-8")

# Lưu từ điển 89 đặc trưng để truy vết nhóm, định nghĩa và cách biến đổi.
feature_descriptions = {
    "avg_price_last_observed": ("Giá", "Giá trung bình gần nhất từ tuần đã hoàn tất trước tuần dự báo."),
    "sales_lag_1": ("Doanh thu trễ", "Doanh thu của sản phẩm ở tuần liền trước."),
    "sales_lag_2": ("Doanh thu trễ", "Doanh thu của sản phẩm cách tuần dự báo hai tuần."),
    "sales_lag_4": ("Doanh thu trễ", "Doanh thu của sản phẩm cách tuần dự báo bốn tuần."),
    "sales_rolling_mean_4": ("Thống kê trượt", "Doanh thu trung bình tối đa bốn tuần đã hoàn tất gần nhất."),
    "sales_rolling_std_4": ("Thống kê trượt", "Độ lệch chuẩn doanh thu tối đa bốn tuần đã hoàn tất gần nhất."),
    "units_lag_1": ("Sản lượng", "Số đơn vị sản phẩm bán ở tuần liền trước."),
    "product_age_weeks": ("Vòng đời", "Số tuần kể từ tuần đầu tiên sản phẩm phát sinh giao dịch."),
    "reviews_count_prior": ("Đánh giá", "Số đánh giá được tạo trước tuần dự báo."),
    "rating_mean_prior": ("Đánh giá", "Điểm đánh giá trung bình chỉ từ phản hồi trước tuần dự báo."),
    "cumulative_product_sales_prior": ("Lịch sử doanh thu", "Doanh thu tích lũy của sản phẩm đến hết tuần trước."),
    "purchase_frequency_prior": ("Hành vi mua", "Tổng số đơn hàng có sản phẩm đến hết tuần trước."),
    "geographic_dispersion_prior": ("Địa lý", "Số bang khách hàng tích lũy đến hết tuần trước."),
    "month": ("Mùa vụ", "Tháng lịch của tuần dự báo."),
    "week_of_year": ("Mùa vụ", "Tuần ISO trong năm của tuần dự báo."),
    "is_peak_shopping_season": ("Mùa vụ", "Cờ mùa mua sắm cao điểm tháng 11 hoặc tháng 12."),
}
schema_rows = []
for feature_name in feature_names:
    if feature_name.startswith("numerical__"):
        source_name = feature_name.removeprefix("numerical__")
        group_name, description = feature_descriptions[source_name]
        schema_rows.append({"feature_name": feature_name, "group": group_name, "description": description, "encoding": "Median imputation, then Z-score scaling fitted on train only"})
    else:
        category_name = feature_name.removeprefix("categorical__category_")
        schema_rows.append({"feature_name": feature_name, "group": "Danh mục sản phẩm", "description": f"Cờ one-hot cho danh mục {category_name}.", "encoding": "Train-mode imputation; fixed 73-category one-hot schema; unknown labels map to all zeros"})
feature_schema = pd.DataFrame(schema_rows)
if len(feature_schema) != 89:
    raise ValueError(f"Expected 89 model features, found {len(feature_schema)}.")
feature_schema.to_csv(config.WEEKLY_FEATURE_SCHEMA_FILE, index=False)

split_summary = pd.DataFrame([
    {
        "split": split_name,
        "rows": len(split_frame),
        "row_share": len(split_frame) / len(featured),
        "week_start": split_frame["week_start"].min(),
        "week_end": split_frame["week_start"].max(),
        "target_sum_brl": split_frame[config.WEEKLY_TARGET_COLUMN].sum(),
        "zero_target_share": split_frame[config.WEEKLY_TARGET_COLUMN].eq(0).mean(),
    }
    for split_name, split_frame in splits.items()
])
split_summary.to_csv(config.WEEKLY_SPLIT_SUMMARY_FILE, index=False)
joblib.dump(preprocessor, config.NOTEBOOK_SAVED_MODELS_DIR / "weekly_preprocessor.joblib")

print(f"Train matrix: {X_train.shape}; validation: {X_validation.shape}; test: {X_test.shape}")
print(f"Encoded predictor count: {len(feature_names)}")
display(split_summary)
print(f"Processed artifacts saved under: {config.PROCESSED_DATA_DIR}")